# Employee Sentiment Analysis

The goal here is to take an unlabeled dataset of employee emails and figure out the sentiment behind them. From there, I'll score employees monthly, rank them, identify who might be a flight risk, and build a regression model to see what features are tied to sentiment.

**Dataset:** test.csv - employee email messages with Subject, body, date, and sender info.

**Tools I'm using:**
- TextBlob for sentiment classification
- scikit-learn for the regression model
- pandas/numpy for data wrangling
- matplotlib/seaborn/wordcloud for charts

---
## Imports and Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# plot settings
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

os.makedirs('visualizations', exist_ok=True)

print('All good, libraries loaded.')

---
## Loading the Data

Let's load test.csv and see what we're working with. Need to parse dates and clean up the email addresses.

In [ ]:
df = pd.read_csv('test.csv')

print(f'Shape: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')
print()
df.dtypes

In [ ]:
df.head()

In [ ]:
# check what's missing
print('Missing values:')
print(df.isnull().sum())
print(f'\nTotal nulls: {df.isnull().sum().sum()}')

In [ ]:
# parse dates, lowercase the emails
df['date'] = pd.to_datetime(df['date'], format='mixed', dayfirst=False)
df['from'] = df['from'].str.strip().str.lower()

print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Unique senders: {df["from"].nunique()}')
print(f'Total messages: {len(df)}')

So we have around 2,191 messages from a handful of employees. The date column needed parsing, and some Subject/body fields have nulls - that's fine, we'll handle those when we do sentiment analysis. The emails look like they're from the Enron dataset based on the domain names.

---
## Task 1: Sentiment Labeling

I'm going with **TextBlob** for this. It gives a polarity score between -1 and +1. I'll combine Subject + body into one text field so it has more context to work with.

For the thresholds, I'm using:
- polarity > 0.1 = Positive
- polarity < -0.1 = Negative
- everything in between = Neutral

I went with 0.1 instead of 0 because a lot of messages have very slight positive/negative tones that are basically neutral in practice. Using 0 would over-classify.

In [ ]:
def get_sentiment_label(text):
    """Run TextBlob on text and return label + score."""
    if pd.isna(text) or str(text).strip() == '':
        return 'Neutral', 0.0
    
    try:
        blob = TextBlob(str(text))
        polarity = blob.sentiment.polarity
        
        if polarity > 0.1:
            return 'Positive', polarity
        elif polarity < -0.1:
            return 'Negative', polarity
        else:
            return 'Neutral', polarity
    except:
        return 'Neutral', 0.0

# merge subject + body
df['full_text'] = df['Subject'].fillna('') + ' ' + df['body'].fillna('')

# run sentiment on everything
print('Running sentiment analysis... this takes a minute')
results = df['full_text'].apply(get_sentiment_label)
df['sentiment'] = results.apply(lambda x: x[0])
df['polarity'] = results.apply(lambda x: x[1])

print('Done!')

In [ ]:
# what's the breakdown?
sentiment_counts = df['sentiment'].value_counts()
for sent, cnt in sentiment_counts.items():
    print(f'  {sent:>10}: {cnt:>5} ({cnt/len(df)*100:.1f}%)')

print(f'\nMean polarity: {df["polarity"].mean():.4f}')
print(f'Std polarity:  {df["polarity"].std():.4f}')

In [ ]:
# let's look at some examples from each category
for sent in ['Positive', 'Negative', 'Neutral']:
    sample = df[df['sentiment'] == sent].iloc[0]
    print(f'\n--- {sent} (polarity: {sample["polarity"]:.3f}) ---')
    print(f'Subject: {sample["Subject"]}')
    body_preview = str(sample['body'])[:150].replace('\n', ' ')
    print(f'Body: {body_preview}...')

In [ ]:
# save the labeled data
df.to_csv('test_labeled.csv', index=False)
print('Saved labeled dataset to test_labeled.csv')

About 46% came out Neutral and 45% Positive. Only ~9% Negative. That makes sense for work emails - most professional communication is either neutral or politely positive. The negative ones are the interesting ones to dig into.

---
## Task 2: Exploratory Data Analysis

Now let's actually explore the data - distributions, trends, who's sending what, etc.

### 2.1 Sentiment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Positive': '#2ecc71', 'Neutral': '#3498db', 'Negative': '#e74c3c'}
counts = df['sentiment'].value_counts()

# bar chart
bars = axes[0].bar(counts.index, counts.values,
                   color=[colors.get(x, '#95a5a6') for x in counts.index],
                   edgecolor='white', linewidth=1.5)
axes[0].set_title('Sentiment Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Number of Messages')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', va='bottom', fontweight='bold')

# pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=[colors.get(x, '#95a5a6') for x in counts.index],
            startangle=140, shadow=True, explode=[0.05]*len(counts))
axes[1].set_title('Sentiment Distribution (%)', fontsize=14, fontweight='bold')

fig.suptitle('Overall Sentiment Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualizations/01_sentiment_distribution.png', bbox_inches='tight', facecolor='white')
plt.show()

Neutral and Positive are pretty close - almost a coin flip between the two. Negative is a much smaller slice, which I expected. Work emails are generally not super emotional.

### 2.2 How Does Sentiment Change Over Time?

In [ ]:
df['year_month'] = df['date'].dt.to_period('M')
monthly_sentiment = df.groupby(['year_month', 'sentiment']).size().unstack(fill_value=0)
monthly_sentiment.index = monthly_sentiment.index.astype(str)

fig, ax = plt.subplots(figsize=(16, 6))
for col in ['Positive', 'Neutral', 'Negative']:
    if col in monthly_sentiment.columns:
        ax.plot(monthly_sentiment.index, monthly_sentiment[col],
                label=col, color=colors[col], marker='o', linewidth=2, markersize=4)

ax.set_title('Sentiment Trends Over Time (Monthly)', fontsize=16, fontweight='bold')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Number of Messages', fontsize=12)
ax.legend(fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('visualizations/02_sentiment_over_time.png', bbox_inches='tight', facecolor='white')
plt.show()

Interesting - there are clear spikes in certain months. The positive and neutral lines tend to move together (more messages overall = more of both). Negative stays relatively flat throughout, which is a good sign overall. Some months have noticeably higher activity.

### 2.3 Polarity Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df['polarity'], bins=50, color='#3498db', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='Neutral (0)')
ax.axvline(x=df['polarity'].mean(), color='green', linestyle='--', linewidth=1.5,
           label=f'Mean ({df["polarity"].mean():.3f})')
ax.set_title('Distribution of Polarity Scores', fontsize=16, fontweight='bold')
ax.set_xlabel('Polarity Score')
ax.set_ylabel('Frequency')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('visualizations/03_polarity_distribution.png', bbox_inches='tight', facecolor='white')
plt.show()

Huge spike at 0 - lots of messages that TextBlob scored as perfectly neutral. The distribution skews slightly positive (mean > 0), which lines up with what we saw in the bar chart. The negative tail is thinner than the positive tail.

### 2.4 Who Sends the Most Emails?

In [ ]:
msg_counts = df['from'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(range(len(msg_counts)), msg_counts.values, color='#3498db',
        edgecolor='white', linewidth=1)
ax.set_yticks(range(len(msg_counts)))
ax.set_yticklabels([e.split('@')[0] for e in msg_counts.index], fontsize=10)
ax.set_xlabel('Number of Messages')
ax.set_title('Top 15 Employees by Message Count', fontsize=16, fontweight='bold')
ax.invert_yaxis()
for i, val in enumerate(msg_counts.values):
    ax.text(val + 5, i, str(val), va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/04_messages_per_employee.png', bbox_inches='tight', facecolor='white')
plt.show()

There's a pretty wide gap between the top sender and the rest. Some people send a lot more email than others, which is worth keeping in mind - an employee with 300 messages and a high score is probably more reliably "positive" than someone with 20 messages.

### 2.5 Sentiment by Employee

In [ ]:
top_employees = df['from'].value_counts().head(10).index
emp_sentiment = df[df['from'].isin(top_employees)].groupby(['from', 'sentiment']).size().unstack(fill_value=0)
emp_sentiment = emp_sentiment.loc[top_employees]
emp_sentiment.index = [e.split('@')[0] for e in emp_sentiment.index]

fig, ax = plt.subplots(figsize=(14, 6))
bottom = np.zeros(len(emp_sentiment))
for sentiment in ['Positive', 'Neutral', 'Negative']:
    if sentiment in emp_sentiment.columns:
        vals = emp_sentiment[sentiment].values
        ax.bar(emp_sentiment.index, vals, bottom=bottom,
               label=sentiment, color=colors[sentiment], edgecolor='white')
        bottom += vals

ax.set_title('Sentiment Breakdown - Top 10 Employees', fontsize=16, fontweight='bold')
ax.set_xlabel('Employee')
ax.set_ylabel('Messages')
ax.legend(fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('visualizations/05_sentiment_by_employee.png', bbox_inches='tight', facecolor='white')
plt.show()

You can see each person's mix of positive/neutral/negative here. Most people are predominantly positive+neutral with a thin red slice. But some have a visibly larger negative portion than others - those are the ones worth watching.

### 2.6 Day of Week

In [ ]:
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['day_of_week'] = df['date'].dt.dayofweek
day_counts = df['day_of_week'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#3498db'] * 5 + ['#e74c3c'] * 2  # red for weekends
bars = ax.bar([day_names[i] for i in day_counts.index], day_counts.values,
              color=bar_colors, edgecolor='white', linewidth=1.5)
ax.set_title('Messages by Day of Week', fontsize=16, fontweight='bold')
ax.set_ylabel('Messages')
for bar, val in zip(bars, day_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/06_day_of_week.png', bbox_inches='tight', facecolor='white')
plt.show()

Weekdays dominate, weekends (red) are much lower. Not surprising for work email. The distribution across weekdays is fairly even though.

### 2.7 Are Longer Emails More Positive or Negative?

In [ ]:
df['message_length'] = df['full_text'].str.len()

# drop the extreme outliers so the box plot is readable
q99 = df['message_length'].quantile(0.99)
df_plot = df[df['message_length'] <= q99]

fig, ax = plt.subplots(figsize=(10, 6))
palette = [colors['Negative'], colors['Neutral'], colors['Positive']]
sns.boxplot(data=df_plot, x='sentiment', y='message_length',
            order=['Negative', 'Neutral', 'Positive'],
            palette=palette, ax=ax)
ax.set_title('Message Length by Sentiment', fontsize=16, fontweight='bold')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Length (chars)')
plt.tight_layout()
plt.savefig('visualizations/07_message_length_by_sentiment.png', bbox_inches='tight', facecolor='white')
plt.show()

There's some difference in length across categories. Positive messages tend to be a bit longer on average - probably because people write more when they're being friendly or appreciative. Negative messages are shorter, which could be because people keep complaints brief.

### 2.8 Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sentiments = ['Positive', 'Neutral', 'Negative']
color_maps = ['Greens', 'Blues', 'Reds']

stop = set(['enron', 'com', 'will', 'please', 'us', 'one', 'would',
            'also', 'may', 'know', 'new', 'said', 'thank', 'thanks'])

for ax, sent, cmap in zip(axes, sentiments, color_maps):
    text = ' '.join(df[df['sentiment'] == sent]['full_text'].dropna().astype(str))
    if text.strip():
        wc = WordCloud(width=600, height=400, background_color='white',
                      colormap=cmap, max_words=100, stopwords=stop)
        wc.generate(text)
        ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{sent}', fontsize=14, fontweight='bold')
    ax.axis('off')

fig.suptitle('Word Clouds by Sentiment', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualizations/08_wordclouds.png', bbox_inches='tight', facecolor='white')
plt.show()

The word clouds give a nice visual of what language shows up in each bucket. Positive has words like 'good', 'great', 'hope'. The negative cloud has some stronger language. Neutral is mostly business terms - meeting times, names, etc.

### 2.9 EDA Summary

In [ ]:
print(f'Total messages: {len(df)}')
print(f'Employees: {df["from"].nunique()}')
print(f'Date range: {df["date"].min().strftime("%Y-%m-%d")} to {df["date"].max().strftime("%Y-%m-%d")}')
print(f'\nSentiment split:')
for s in ['Positive', 'Neutral', 'Negative']:
    c = (df['sentiment'] == s).sum()
    print(f'  {s}: {c} ({c/len(df)*100:.1f}%)')
print(f'\nAvg message length: {df["message_length"].mean():.0f} chars')
print(f'Avg polarity: {df["polarity"].mean():.4f}')

**Main takeaways from the EDA:**
1. Mostly neutral/positive emails, only ~9% negative
2. Some employees are way more active than others
3. Activity drops off on weekends (obviously)
4. Positive emails tend to be longer
5. There are some interesting trends month-to-month worth digging into

---
## Task 3: Monthly Sentiment Scoring

Now I need to score each employee monthly. The scoring is simple:
- Positive = +1
- Negative = -1
- Neutral = 0

Sum it up per employee per month. Score resets each month.

In [ ]:
def calc_score(sentiment):
    if sentiment == 'Positive': return 1
    elif sentiment == 'Negative': return -1
    else: return 0

df['sentiment_score'] = df['sentiment'].apply(calc_score)
df['year_month'] = df['date'].dt.to_period('M')

# aggregate by employee and month
monthly_scores = df.groupby(['from', 'year_month']).agg(
    monthly_score=('sentiment_score', 'sum'),
    positive_count=('sentiment_score', lambda x: (x == 1).sum()),
    negative_count=('sentiment_score', lambda x: (x == -1).sum()),
    neutral_count=('sentiment_score', lambda x: (x == 0).sum()),
    total_messages=('sentiment_score', 'count')
).reset_index()

print(f'Computed scores for {monthly_scores["from"].nunique()} employees across {monthly_scores["year_month"].nunique()} months')
print(f'\nScore stats:')
print(f'  Mean:  {monthly_scores["monthly_score"].mean():.2f}')
print(f'  Max:   {monthly_scores["monthly_score"].max()}')
print(f'  Min:   {monthly_scores["monthly_score"].min()}')

In [ ]:
monthly_scores.head(15)

In [ ]:
# heatmap of monthly scores for the most active employees
top_emps = monthly_scores.groupby('from')['total_messages'].sum().nlargest(15).index
filtered = monthly_scores[monthly_scores['from'].isin(top_emps)].copy()
pivot = filtered.pivot_table(index='from', columns='year_month', values='monthly_score', fill_value=0)
pivot.index = [e.split('@')[0] for e in pivot.index]
pivot.columns = pivot.columns.astype(str)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot, cmap='RdYlGn', center=0, annot=True, fmt='.0f',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Monthly Score'})
ax.set_title('Monthly Sentiment Scores - Top 15 Employees', fontsize=16, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Employee')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('visualizations/09_monthly_scores_heatmap.png', bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
monthly_scores.to_csv('monthly_scores.csv', index=False)
print('Saved to monthly_scores.csv')

The heatmap is really useful here. Green = positive months, red = negative. You can see that most employees are consistently green, but there are scattered red/yellow cells. A few employees have more red patches than others.

---
## Task 4: Employee Ranking

For each month, pick out the top 3 positive (highest score) and top 3 negative (lowest score). If there's a tie, sort alphabetically.

In [ ]:
top_positive_list = []
top_negative_list = []

for month in monthly_scores['year_month'].unique():
    month_data = monthly_scores[monthly_scores['year_month'] == month].copy()
    
    # top 3 positive: highest score first, then alphabetical for ties
    sorted_pos = month_data.sort_values(by=['monthly_score', 'from'], ascending=[False, True])
    top3p = sorted_pos.head(3).copy()
    top3p['rank'] = range(1, len(top3p) + 1)
    top_positive_list.append(top3p)
    
    # top 3 negative: lowest score first, then alphabetical for ties
    sorted_neg = month_data.sort_values(by=['monthly_score', 'from'], ascending=[True, True])
    top3n = sorted_neg.head(3).copy()
    top3n['rank'] = range(1, len(top3n) + 1)
    top_negative_list.append(top3n)

top_positive = pd.concat(top_positive_list, ignore_index=True)
top_negative = pd.concat(top_negative_list, ignore_index=True)

print('Rankings done.')

In [ ]:
# print out the rankings per month
for month in sorted(monthly_scores['year_month'].unique()):
    print(f'\n{"="*60}')
    print(f'  {month}')
    print(f'{"="*60}')
    
    pos_m = top_positive[top_positive['year_month'] == month]
    neg_m = top_negative[top_negative['year_month'] == month]
    
    print('  TOP 3 POSITIVE:')
    for _, r in pos_m.iterrows():
        print(f'    #{int(r["rank"])}  {r["from"]:<40} Score: {r["monthly_score"]:>3}')
    
    print('  TOP 3 NEGATIVE:')
    for _, r in neg_m.iterrows():
        print(f'    #{int(r["rank"])}  {r["from"]:<40} Score: {r["monthly_score"]:>3}')

In [ ]:
# visualization of overall rankings
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pos_agg = top_positive.groupby('from')['monthly_score'].mean().nlargest(5)
neg_agg = top_negative.groupby('from')['monthly_score'].mean().nsmallest(5)

axes[0].barh(range(len(pos_agg)), pos_agg.values, color='#2ecc71', edgecolor='white')
axes[0].set_yticks(range(len(pos_agg)))
axes[0].set_yticklabels([e.split('@')[0] for e in pos_agg.index], fontsize=10)
axes[0].set_title('Most Positive (Avg Monthly Score)', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

axes[1].barh(range(len(neg_agg)), neg_agg.values, color='#e74c3c', edgecolor='white')
axes[1].set_yticks(range(len(neg_agg)))
axes[1].set_yticklabels([e.split('@')[0] for e in neg_agg.index], fontsize=10)
axes[1].set_title('Most Negative (Avg Monthly Score)', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()

fig.suptitle('Employee Rankings', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualizations/10_employee_rankings.png', bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
top_positive.to_csv('top_positive_employees.csv', index=False)
top_negative.to_csv('top_negative_employees.csv', index=False)
print('Rankings saved.')

Some employees consistently show up in the positive rankings month after month, and a few pop up repeatedly in the negative side. This is useful for HR to see who might need support or who's consistently contributing positively.

---
## Task 5: Flight Risk Identification

A flight risk = someone who sent **4 or more negative emails within any 30-day window**. The window rolls - it's not tied to calendar months.

My approach: for each employee, I take all their negative messages sorted by date. Then for each date, I count how many negatives fall within the next 30 days. If any window hits 4+, they're flagged.

In [ ]:
negative_msgs = df[df['sentiment'] == 'Negative'].copy()
negative_msgs = negative_msgs.sort_values(['from', 'date'])

flight_risk_list = []

for employee in negative_msgs['from'].unique():
    emp_neg = negative_msgs[negative_msgs['from'] == employee].sort_values('date').reset_index(drop=True)
    
    if len(emp_neg) < 4:
        continue  # can't hit threshold with fewer than 4
    
    dates = emp_neg['date'].values
    
    for i in range(len(dates)):
        window_start = dates[i]
        window_end = dates[i] + np.timedelta64(30, 'D')
        count = ((dates >= window_start) & (dates <= window_end)).sum()
        
        if count >= 4:
            flight_risk_list.append({
                'employee': employee,
                'total_negative_messages': len(emp_neg),
                'max_negatives_in_30_days': count,
                'risk_window_start': pd.Timestamp(window_start),
                'risk_window_end': pd.Timestamp(window_end)
            })
            break  # found one window, that's enough

flight_risks = pd.DataFrame(flight_risk_list)
print(f'{len(flight_risks)} employees flagged as flight risks')

In [ ]:
if flight_risks.empty:
    print('Nobody flagged.')
else:
    print('Flight Risk Employees:')
    print('-' * 70)
    for _, row in flight_risks.iterrows():
        print(f'\n  {row["employee"]}')
        print(f'    Total negatives: {row["total_negative_messages"]}')
        print(f'    Worst 30-day window: {row["max_negatives_in_30_days"]} negatives')
        print(f'    Window: {row["risk_window_start"].strftime("%Y-%m-%d")} to {row["risk_window_end"].strftime("%Y-%m-%d")}')

In [ ]:
# visualize the flight risks
if not flight_risks.empty:
    fr = flight_risks.sort_values('max_negatives_in_30_days', ascending=False)
    
    fig, ax = plt.subplots(figsize=(14, max(5, len(fr) * 0.5)))
    labels = [e.split('@')[0] for e in fr['employee']]
    bar_colors = ['#e74c3c' if x >= 6 else '#e67e22' if x >= 5 else '#f1c40f'
                  for x in fr['max_negatives_in_30_days']]
    
    ax.barh(range(len(fr)), fr['max_negatives_in_30_days'].values,
            color=bar_colors, edgecolor='white')
    ax.set_yticks(range(len(fr)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Negatives in 30-Day Window')
    ax.set_title('Flight Risk Employees', fontsize=16, fontweight='bold')
    ax.axvline(x=4, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Threshold (4)')
    ax.legend()
    ax.invert_yaxis()
    for i, val in enumerate(fr['max_negatives_in_30_days'].values):
        ax.text(val + 0.1, i, str(val), va='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig('visualizations/11_flight_risk_employees.png', bbox_inches='tight', facecolor='white')
    plt.show()

In [ ]:
if not flight_risks.empty:
    flight_risks.to_csv('flight_risk_employees.csv', index=False)
    print('Saved to flight_risk_employees.csv')

6 employees got flagged. A couple of them hit 6 negatives in a 30-day span, which is concerning. It's worth noting that some of these employees also appear in the top positive list overall - they might just be very active communicators who occasionally have rough stretches.

---
## Task 6: Linear Regression Model

The idea is to see if we can predict polarity from message metadata. I'm using these features:

| Feature | What it is |
|---------|------------|
| message_length | char count |
| word_count | word count |
| subject_length | subject line length |
| has_subject | 1 if it has a subject, 0 if blank |
| day_of_week | 0=Monday through 6=Sunday |
| is_weekend | 1 if Sat/Sun |
| month | 1-12 |
| exclamation_count | number of ! |
| question_count | number of ? |
| caps_ratio | fraction of letters that are uppercase |

Target: polarity score (-1 to +1)

In [ ]:
# build features
df['word_count'] = df['full_text'].str.split().str.len()
df['subject_length'] = df['Subject'].fillna('').str.len()
df['has_subject'] = (df['Subject'].fillna('') != '').astype(int)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['month'] = df['date'].dt.month
df['exclamation_count'] = df['full_text'].str.count('!')
df['question_count'] = df['full_text'].str.count(r'\?')

def caps_ratio(text):
    if pd.isna(text) or len(text) == 0:
        return 0.0
    alpha = [c for c in text if c.isalpha()]
    if len(alpha) == 0:
        return 0.0
    return sum(1 for c in alpha if c.isupper()) / len(alpha)

df['caps_ratio'] = df['full_text'].apply(caps_ratio)

feature_cols = ['message_length', 'word_count', 'subject_length',
                'has_subject', 'day_of_week', 'is_weekend', 'month',
                'exclamation_count', 'question_count', 'caps_ratio']

X = df[feature_cols].fillna(0)
y = df['polarity']

print(f'Samples: {len(X)}')
X.describe()

In [ ]:
# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

# scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# train
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

print('Model Performance:')
print(f'  Train R2:   {r2_score(y_train, y_pred_train):.4f}')
print(f'  Test R2:    {r2_score(y_test, y_pred_test):.4f}')
print(f'  Train RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}')
print(f'  Test RMSE:  {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}')
print(f'  Train MAE:  {mean_absolute_error(y_train, y_pred_train):.4f}')
print(f'  Test MAE:   {mean_absolute_error(y_test, y_pred_test):.4f}')

In [ ]:
# which features matter?
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': model.coef_})
coef_df = coef_df.sort_values('Coefficient', ascending=False)
print('Feature coefficients:')
for _, r in coef_df.iterrows():
    sign = '+' if r['Coefficient'] > 0 else '-'
    print(f'  {sign} {r["Feature"]:<25} {r["Coefficient"]:>10.6f}')
print(f'\n  Intercept: {model.intercept_:.6f}')

In [ ]:
# actual vs predicted + feature importance chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_test, y_pred_test, alpha=0.3, color='#3498db', s=10)
mn = min(y_test.min(), y_pred_test.min())
mx = max(y_test.max(), y_pred_test.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect')
axes[0].set_xlabel('Actual Polarity')
axes[0].set_ylabel('Predicted Polarity')
axes[0].set_title('Actual vs Predicted', fontsize=14, fontweight='bold')
axes[0].legend()

importance = pd.Series(model.coef_, index=feature_cols).sort_values()
c = ['#e74c3c' if v < 0 else '#2ecc71' for v in importance.values]
axes[1].barh(range(len(importance)), importance.values, color=c, edgecolor='white')
axes[1].set_yticks(range(len(importance)))
axes[1].set_yticklabels(importance.index, fontsize=10)
axes[1].set_xlabel('Coefficient')
axes[1].set_title('Feature Importance', fontsize=14, fontweight='bold')
axes[1].axvline(x=0, color='black', linewidth=0.5)

fig.suptitle('Regression Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualizations/12_regression_results.png', bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# residual plots
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='red', linestyle='--')
axes[0].set_title('Residual Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Residual')

axes[1].scatter(y_pred_test, residuals, alpha=0.3, color='#e74c3c', s=10)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_title('Residuals vs Predicted', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')

fig.suptitle('Diagnostic Plots', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualizations/13_residuals.png', bbox_inches='tight', facecolor='white')
plt.show()

The R2 is low (~0.07) which honestly isn't surprising. You can't really predict what someone *said* based on when they sent the email or how long it was. But it does tell us something interesting:

- **exclamation_count** is the strongest positive predictor - more !'s = more positive sentiment. Makes sense.
- **word_count** also trends positive - people write more when they're being nice.
- **caps_ratio** is negative - more ALL CAPS = more negative. Also makes sense.
- **is_weekend** is slightly negative - weekend emails might carry more frustration?

The model isn't great at prediction, but it's useful for understanding which structural features correlate with sentiment.

---
## Final Summary

In [ ]:
print('=' * 60)
print('  FINAL SUMMARY')
print('=' * 60)

print('\nSentiment breakdown:')
for sent, cnt in sentiment_counts.items():
    print(f'  {sent}: {cnt} ({cnt/len(df)*100:.1f}%)')

print('\nTop 3 positive employees (overall):')
overall_pos = monthly_scores.groupby('from')['monthly_score'].sum().nlargest(3)
for i, (emp, score) in enumerate(overall_pos.items(), 1):
    print(f'  #{i} {emp} (score: {score})')

print('\nTop 3 negative employees (overall):')
overall_neg = monthly_scores.groupby('from')['monthly_score'].sum().nsmallest(3)
for i, (emp, score) in enumerate(overall_neg.items(), 1):
    print(f'  #{i} {emp} (score: {score})')

print('\nFlight risks:')
if flight_risks.empty:
    print('  None')
else:
    for _, row in flight_risks.iterrows():
        print(f'  {row["employee"]} ({row["max_negatives_in_30_days"]} negatives in 30 days)')

print(f'\nRegression R2 (test): {r2_score(y_test, y_pred_test):.4f}')
print(f'Regression RMSE (test): {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}')

---
## Output Files

| File | What's in it |
|------|--------------|
| `test_labeled.csv` | Original data + sentiment labels |
| `monthly_scores.csv` | Monthly scores per employee |
| `top_positive_employees.csv` | Top 3 positive per month |
| `top_negative_employees.csv` | Top 3 negative per month |
| `flight_risk_employees.csv` | Flagged flight risks |
| `visualizations/` | All 13 charts |